# 03. Serving the fine-tuned model with Ray Serve

**AI Drug Discovery & Development Summit 2026 · Hands-on workshop · Part 3 of 3**

**Workshop series:** [01 Ray and offline inference](01_ray_and_offline_inference.ipynb) · [02 Train and tune](02_train_and_tune.ipynb) · [03 Serve](03_serve.ipynb)

Notebook 02 made a better model for one target. Now chemists get to use it, from the 3D viewer page or
from Python. We also add new model versions while the old ones keep working.

| § | Topic | Ray library | Min |
|---|---|---|---|
| 0 | Setup | | 2 |
| 6 | Serving the fine-tuned model | Ray Serve | 17 |
| 7 | Summary and next steps | | 5 |

Section numbers continue from the earlier notebooks, so this one starts at 6.


## 0. Setup

- The Workspace image has Ray, Boltz-2 and two OpenFold3 versions (0.5 and 0.4.3).
- This notebook runs on a CPU machine. GPU machines (L4) start when a cell needs one and stop when idle.
- Data, model weights and results are on shared storage.

In [ ]:
import os, sys, json, time, warnings, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import ray

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

from aiddd_pipeline import (
    SHARED,
    TARGETS,
    BOLTZ_CACHE,
    OF3_OB_CKPT,
    OF3_P2_CKPT,
    OF3_FT_CKPT,
    OF3_OURS_CKPT,
    OF3_LEGACY,
    Boltz2Predictor,
    OF3Predictor,
    load_ligands,
    validate,
    make_scorer,
)
import workshop_helpers as wh

# True: run section 6 on your own GPUs. False: use the shared Service, like the rest of the room.
USE_GPUS = False

# Data is downloaded once, ahead of the workshop (README step 2). The weights (20 GB) are read
# only when section 6 runs on your own GPUs; against the shared Service they are never loaded here.
needed = [TARGETS[t].path("metadata") for t in wh.TARGET_KEYS]
if USE_GPUS:
    needed += [
        BOLTZ_CACHE,
        OF3_OB_CKPT,
        OF3_P2_CKPT,
        OF3_FT_CKPT,
        OF3_OURS_CKPT,
    ]
missing = [str(p) for p in needed if not p.exists()]
assert (
    not missing
), f"Missing {missing}. Run `python scripts/setup_data.py` and `python scripts/prepare_mpro.py` first."

# True: 6.4 and 6.5 change the shared Service (the presenter only)
PRESENTER = False
SERVICE_NAME = (
    "aiddd-viewer"  # the shared Service: viewer page and predictions at one URL
)
GPU = "L4"  # GPU type for every GPU step in this notebook

# The Service's URL: from AIDDD_SERVICE_URL, or looked up by name
SERVICE_URL = os.environ.get("AIDDD_SERVICE_URL", "")
if not SERVICE_URL:
    try:
        import anyscale

        SERVICE_URL = anyscale.service.status(name=SERVICE_NAME).query_url or ""
    except Exception:  # no access to the Service from this workspace
        pass

wh.MINE.mkdir(parents=True, exist_ok=True)
print(f"shared data:  {SHARED}")
print(f"your outputs: {wh.MINE}")
print(
    f"viewer:       {SERVICE_URL or 'not found. Set AIDDD_SERVICE_URL to the URL the presenter shares, then re-run Setup.'}"
)

try:
    ray.init(
        runtime_env={"env_vars": {"AIDDD_SHARED": str(SHARED)}},
        ignore_reinit_error=True,
        logging_level="ERROR",
    )
except ValueError:  # inside an Anyscale Job, which already sets the environment
    ray.init(ignore_reinit_error=True, logging_level="ERROR")
dashboard = (
    os.environ.get("RAY_OVERRIDE_DASHBOARD_URL")
    or ray.get_runtime_context().dashboard_url
)
print(
    f"Ray Dashboard metrics: {dashboard.rstrip('/')}/#/metrics   (open in a new browser tab)"
)

In [ ]:
# Context from notebooks 01 and 02, loaded here so this notebook stands alone
results = {t: wh.load_results(t) for t in wh.TARGET_KEYS}
ev71 = results["ev71-2a"]
x7259a = ev71[ev71.complex_name == "A71EV2A-x7259a"].iloc[
    0
]  # the late follow-on used in 6.3 to 6.5
print("loaded benchmark results:", ", ".join(results))

## 6. Serving with Ray Serve: one base model, many specialists

⏱ **Budget: 17 minutes.**

**Why.** A fine-tuned model is a file until a chemist can use it. A chemist with an idea for the next
compound wants to see where a model puts it in the pocket before spending days to weeks making it. Other projects keep using the general model from the same service.

**What this section shows.**
- **A.** Many models behind one address, each added with no restart (6.1 to 6.5). Ray Serve **model
  multiplexing**: one deployment serves the general **base model** and every **specialist** (a model
  fine-tuned for one target), and each request names the model it wants with a **model ID**.
- **B.** CPU steps and GPU models composed in one Python app (6.6).
- **C.** GPUs from any zone in the region (6.6 and 6.8).

The whole room uses one shared Anyscale Service. Only the presenter changes it.

| Step | What we do | What you see |
|---|---|---|
| Start | The presenter sets the Service to step 1: base model only | Setup prints the Service URL; the viewer page uses the same one |
| 6.1 | Read the multiplexed deployment's code | One deployment, many models |
| 6.2 | Meet the 6 test molecules | Each has a crystal structure: the right answer |
| 6.3 | Ask the base model about x7259a | 5.2 Å from the crystal: wrong |
| 6.4 | Add our fine-tune, `ev71-v1`, with no restart | 1.0 Å: right |
| 6.5 | Add the public fine-tune, `ev71-v2`, with no restart | 0.4 Å |
| 6.6 | See how the Service is built | CPU steps around two GPU models |
| 6.7 | The whole room sends requests at once | Every answer in under a second |
| 6.8 | What comes next | Scaling, more models, GPUs from more zones |

![The Service you will call: one request, composed across CPU and GPU deployments](docs/images/serve_composition.svg)

This is the Service your requests go to from 6.3 on. Your request goes to the Viewer on CPU, which asks
two GPU models at the same time and sends one answer back. The green box (OF3Pool) is what 6.1 to 6.5
cover; 6.6 walks the arrows.

Every pose was computed once before the session and saved, so your requests come back at once and use no GPU.


In [ ]:
# Presenter only: create the Service if it does not exist (5 to 10 min, once), then set it to
# step 1, base model only (about 30 s, no restart)
if PRESENTER:
    !anyscale service status -n aiddd-viewer > /dev/null 2>&1 || anyscale service deploy -f jobs/service.yaml
    !anyscale service wait -n aiddd-viewer --state RUNNING
    !anyscale service deploy -f jobs/demo_1_base.yaml --in-place

### 6.1 A multiplexed deployment

**Ray Serve words.**
- A **deployment** is a Python class that Serve runs for you.
- A **replica** is one running copy of it. Each replica gets the CPUs, GPUs and memory it asks for, and
  Serve adds or removes replicas as traffic changes.
- An **application** is one or more deployments that work together. You call it over HTTP, or from
  Python with a **handle**.
- On Anyscale, a **Service** runs applications at one fixed URL and updates them with no downtime.

**Standard versus multiplexed.** A standard deployment holds one model, loaded once when the replica
starts. Ten models means ten deployments, each with its own GPUs, and most of them sit idle. A
**multiplexed** deployment holds many models. Each request names a model ID. A replica loads that model
the first time someone asks for it and keeps it, and Serve sends later requests for that ID to a replica
that has it. A new model is a new ID, not a new deployment.

![A standard Ray Serve deployment next to a multiplexed one](docs/images/serve_multiplexing.svg)

**The code.** `@serve.multiplexed` does the work.
- `__call__` asks Serve which model the request wants: `serve.get_multiplexed_model_id()`.
- `load` turns that ID into a model: it builds OpenFold3 and reads the checkpoint, once (10 to 70 s).
  Serve keeps the result.
- `max_num_models_per_replica=3` keeps up to 3 models. A 4th pushes out the one used least recently.
- `user_config` is the list of models the deployment can serve. Nothing loads until someone asks. To add a
  model to a running Service, you change this list: no restart.

The shared Service runs this same deployment as `OF3Pool` (6.6): one replica, half an L4, up to 3 models.

**RAM, not the GPU, limits how many models fit.** A loaded model waits in the replica's normal memory
(RAM), about 3 GB each, and goes to the GPU only during a request. With 3 models, the replica uses about
15 GB of the machine's 32 GB. `"memory": 15e9` tells Ray this, so Ray puts the replica only on a machine
with 15 GB free, and two memory-heavy replicas never share a machine. Ray uses this number only to pick a
machine. It does not limit the replica.

**Two more details.**
- OpenFold3 0.4.x lives in a separate Python in the image. `runtime_env={"py_executable": ...}` gives only
  this deployment that Python.
- Slow work runs in a thread (`asyncio.to_thread`), so the replica still answers Serve's health checks
  while it loads a model.

In [ ]:
import asyncio
from ray import serve
from aiddd_pipeline import (
    LIVE_MODELS,
    OF3_LEGACY_PY,
)  # model id -> {version, name, checkpoint}
from of3_live import LiveModel  # OpenFold3 with its weights loaded, and a pose cache


@serve.deployment(
    ray_actor_options={
        "num_gpus": 0.5,
        "memory": 15e9,  # host memory for 3 loaded models: Ray places the replica where it is free
        "accelerator_type": "L4",
        "runtime_env": {"py_executable": OF3_LEGACY_PY},
    }
)  # OpenFold3 0.4.x
class OF3Pool:
    def reconfigure(self, models):  # {model_id: {version, name, checkpoint}}
        self.models = models

    @serve.multiplexed(
        max_num_models_per_replica=3
    )  # 3 models in memory; a fourth evicts the least used
    async def load(self, model_id):  # builds OpenFold3 and reads the checkpoint, once
        return await asyncio.to_thread(LiveModel, self.models[model_id])

    async def __call__(self, smiles, fresh=False):
        model = await self.load(serve.get_multiplexed_model_id())
        return await asyncio.to_thread(
            model, smiles, fresh
        )  # pose, plus where it came from


models = dict(
    LIVE_MODELS
)  # base, ev71-v1, ev71-v2: registered now, each loaded on its first request
if USE_GPUS:  # the presenter: this application, on the workshop's GPUs
    handle = serve.run(
        OF3Pool.options(user_config=models).bind(),
        name="aiddd-pose",
        route_prefix=None,
    )  # called by handle from this notebook only, no HTTP route
else:  # everyone else: the shared Service runs the same kind of deployment, for the whole room
    handle = None
    print(
        "The presenter runs this deployment on GPUs. Your requests in this section go to the shared Service:",
        SERVICE_URL
        or "not set yet. Set AIDDD_SERVICE_URL (README, Workshop format) and re-run Setup.",
    )

### 6.2 The test molecules

All six are EV-A71 2A compounds made late in the project (follow-ons). Each has a crystal structure, so
we know the right answer. No fine-tune saw them in training.

| Compound | Base model | Our fine-tune (v1) | Public fine-tune (v2) | SMILES |
|---|---|---|---|---|
| **x7259a**, the demo: most potent here (pKD 7.9) | 5.2 Å | 1.0 Å | 0.4 Å | `CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7608a** | 4.8 Å | 0.3 Å | 0.5 Å | `CSc1ncnc(N(C)CC(F)(F)F)c1CC(=O)N1C[C@@H](C(N)=O)C[C@@H]2C[C@@H]21` |
| **x7024a** | 6.1 Å | 1.1 Å | 0.8 Å | `CSc1ncnc(N2CCC3(CCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x6714a** | 4.7 Å | 0.9 Å | 0.4 Å | `NC(=O)[C@H]1CCC(=O)N(c2cc3cnccc3c(F)c2F)C1` |
| **x7011a** | 4.8 Å | 1.4 Å | 0.5 Å | `CC[C@H]1CCCN1c1ncnc(SC)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |
| **x7101a**, harder: only v2 gets it | 6.2 Å | 4.3 Å | 0.8 Å | `CSc1ncnc(N2CCC3(CCCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1` |

The numbers are ligand RMSD: how far the predicted pose is from the crystal. Under 2 Å is correct. Each
is one prediction (seed 42); a new run of the same molecule can differ by about 0.2 Å.

### 6.3 Ask the base model

Each request names its model: `"model": "base"` over HTTP, or `handle.options(multiplexed_model_id="base")`
from Python. The base model never saw this protein pocket with a ligand in it, so it usually gets these
molecules wrong.

Your answer comes from the cache. The presenter's runs live: with `PRESENTER = True`, the cell first
deletes this pose from the cache, so the model really runs (about 25 s, plus the load time if the model
is not in memory yet). Each answer says how it was made.

The last line shows which models the OpenFold3 replica holds in memory now.

**The Service's HTTP API.** Every cell below that talks to the Service makes one of these calls. The
viewer page makes the same ones.

| Method and path | Body | Returns |
|---|---|---|
| `GET /` | | The viewer page |
| `POST /predict` | `{"smiles": "...", "model": "base"}` (`model` is optional: default is the model the viewer uses) | The pose, its RMSD to the crystal, the Boltz-2 affinity, and how it was made |
| `GET /models` | | The models registered, the model the viewer uses, and the models in memory |

The same calls from a terminal:

```bash
curl -X POST https://aiddd-viewer-5ms2e.cld-g54aiirwj1s8t9kt.s.anyscaleuserdata.com/predict \
     -H 'Content-Type: application/json' \
     -d '{"smiles": "CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1", "model": "base"}'
curl https://aiddd-viewer-5ms2e.cld-g54aiirwj1s8t9kt.s.anyscaleuserdata.com/models
```

Each cell prints the request it sends and the response it gets, as JSON. The pose text (`cif`) is cut
to its first line.


In [ ]:
import requests

DEMO_COMPOUNDS = {
    "x7259a": "CSc1ncnc(N2CCC[C@@H]2C2CC2)c1CC(=O)N1CCC[C@H](C(N)=O)C1",
    "x7608a": "CSc1ncnc(N(C)CC(F)(F)F)c1CC(=O)N1C[C@@H](C(N)=O)C[C@@H]2C[C@@H]21",
    "x7024a": "CSc1ncnc(N2CCC3(CCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1",
    "x6714a": "NC(=O)[C@H]1CCC(=O)N(c2cc3cnccc3c(F)c2F)C1",
    "x7011a": "CC[C@H]1CCCN1c1ncnc(SC)c1CC(=O)N1CCC[C@H](C(N)=O)C1",
    "x7101a": "CSc1ncnc(N2CCC3(CCCCC3)C2)c1CC(=O)N1CCC[C@H](C(N)=O)C1",
}
# choose compound to predict
compound = "x7259a"
smiles = DEMO_COMPOUNDS[compound]
ev71 = results["ev71-2a"]
compound_data = ev71[ev71.complex_name == f"A71EV2A-{compound}"].iloc[0]


def show(out, label):
    """Who answered and how, in one line, then the predicted pose (green) over the crystal (orange)."""
    if out.get("status") == "failed":
        if "unknown model id" in out["error"]:
            print(
                f"{label}: not on the Service yet. The presenter adds it in this step; run the cell again after."
            )
        else:
            print(f"{label}: {out['error']}")
        return
    who = f"the Service {SERVICE_NAME}" if "pose_model" in out else "your deployment"
    how = (
        "from its pose cache, no GPU work"
        if out["source"] == "cache"
        else f"live in {out['seconds']:.0f} s"
        + (
            f", {out['load_seconds']:.0f} s of it loading the model"
            if out["load_seconds"]
            else ""
        )
    )
    rmsd = (out.get("answer_key") or {}).get("lig_rmsd")
    print(
        f"{label}: {who} answered with {out['model']['name']}, {how}"
        + (f"; {rmsd:.1f} Å from the crystal" if rmsd is not None else "")
        + (f"; GPU in {out['zone']}" if out.get("zone") else "")
    )
    if (
        out.get("format") == "pdb"
    ):  # the Service sends the pose already moved onto the crystal
        wh.show_aligned(out["cif"], compound_data, label=label, height=340)
    else:  # your deployment sends the raw prediction
        wh.show_analog(out["cif_path"], compound_data, label=label, height=340)


def print_json(body):
    """Print a request or response body as indented JSON, with the pose text cut to its first line."""
    body = {
        k: f"{v.splitlines()[0]} ... ({len(v.splitlines())} lines)" if k == "cif" else v
        for k, v in body.items()
    }
    print(json.dumps(body, indent=2, default=str))


def in_memory():
    """The models the Service's OpenFold3 replica holds in memory now."""
    if not USE_GPUS and SERVICE_URL:
        resident = requests.get(f"{SERVICE_URL}/models", timeout=60).json()["resident"]
        print("in memory on the OpenFold3 replica:", resident)


if (
    PRESENTER
):  # remove this pose from the Service's cache, so the Service computes it live
    requests.post(
        f"{SERVICE_URL}/cache/clear",
        json={"compound": compound, "model": "base"},
        timeout=60,
    )

if (
    USE_GPUS
):  # your own deployment, through its Ray Serve handle (fresh: run the model even if cached)
    out = await handle.options(multiplexed_model_id="base").remote(smiles, fresh=True)
else:  # the shared Anyscale Service, over HTTP, as the viewer page calls it
    request = {"smiles": smiles, "model": "base"}
    print(f"POST {SERVICE_URL}/predict")
    print_json(request)
    out = requests.post(f"{SERVICE_URL}/predict", json=request, timeout=900).json()
print("response:")
print_json(out)
show(out, f"{compound}, base model")
in_memory()

### 6.4 Add the specialist

Notebook 02 fine-tuned the base model on this target's fragment screen. Its model ID is `ev71-v1`. It
loads the first time someone asks for it, next to the base model, which stays loaded for everyone else.

**Why it matters.** Each checkpoint that comes out of training becomes a config change, not a new
endpoint, and the base model keeps serving everyone else while it happens.

The presenter adds it in the next cell with `demo_2_ft.yaml --in-place`: only `user_config` changes, so
Serve calls `reconfigure()` on the running replicas, with no restart and no new machine. The viewer page
then uses it for this target. It answers 10 to 35 s later: if your request says it is not on the Service
yet, run the cell again. In-place updates are meant for this kind of fast iteration; 6.6 covers how a code
change goes to production.

With `PRESENTER = True`, the request cell first deletes `ev71-v1`'s cached pose, so `ev71-v1` runs live
(about 40 s if it has to load, about 25 s if it is already in memory).


In [ ]:
if PRESENTER:  # add our fine-tune to the shared Service, in place: no restart
    !anyscale service wait -n aiddd-viewer --state RUNNING    # the previous update has finished
    !anyscale service deploy -f jobs/demo_2_ft.yaml --in-place
    # Wait until both parts are updated: the model the Viewer uses and OF3Pool's model list.
    # They change in no fixed order, and asking OF3Pool for a model it does not have yet fails
    while True:
        m = requests.get(f"{SERVICE_URL}/models", timeout=10).json()
        if m["route"] == "ev71-v1" and "ev71-v1" in m["registry"]:
            break
        time.sleep(2)  # the new model answers in 10 to 35 s

In [ ]:
if PRESENTER:  # remove this pose from the cache, so ev71-v1 loads and runs live
    requests.post(
        f"{SERVICE_URL}/cache/clear",
        json={"compound": compound, "model": "ev71-v1"},
        timeout=60,
    )

if USE_GPUS:
    out = await handle.options(multiplexed_model_id="ev71-v1").remote(smiles)
else:
    request = {"smiles": smiles, "model": "ev71-v1"}
    print(f"POST {SERVICE_URL}/predict")
    print_json(request)
    out = requests.post(f"{SERVICE_URL}/predict", json=request, timeout=900).json()
print("response:")
print_json(out)
show(out, f"{compound}, our fine-tune")
in_memory()

Same molecule, same address, a different model ID: the pose goes from 5.2 Å to 1.0 Å off the crystal.
One replica holds both models: the in-memory list includes `base` and `ev71-v1`.

### 6.5 A newer version

A better fine-tune arrives: the public OpenBind release, model ID `ev71-v2`. It gets x7259a to 0.4 Å and
also gets x7101a right. `ev71-v1` still answers by its ID, so you can go back to it or compare the two.
Here `ev71-v2` answers from the cache, so it does not load: it is on the model list, but not in memory.
Its first live request would load it. The v2 file is 6.7 GB (it also holds training state), so that
takes 20 to 70 s; after that, a prediction takes the usual 25 s.

The presenter adds it in the next cell with `demo_3_next_ft.yaml --in-place`, the same way as 6.4.


In [ ]:
if PRESENTER:  # add the public fine-tune to the shared Service, in place: no restart
    !anyscale service wait -n aiddd-viewer --state RUNNING    # the previous update has finished
    !anyscale service deploy -f jobs/demo_3_next_ft.yaml --in-place
    # Wait until both parts are updated: the model the Viewer uses and OF3Pool's model list.
    # They change in no fixed order, and asking OF3Pool for a model it does not have yet fails
    while True:
        m = requests.get(f"{SERVICE_URL}/models", timeout=10).json()
        if m["route"] == "ev71-v2" and "ev71-v2" in m["registry"]:
            break
        time.sleep(2)  # the new model answers in 10 to 35 s

if USE_GPUS:
    out = await handle.options(multiplexed_model_id="ev71-v2").remote(smiles)
else:
    request = {"smiles": smiles, "model": "ev71-v2"}
    print(f"POST {SERVICE_URL}/predict")
    print_json(request)
    out = requests.post(f"{SERVICE_URL}/predict", json=request, timeout=900).json()
print("response:")
print_json(out)
show(out, f"{compound}, the public fine-tune")
in_memory()

**No model is pushed out here.** The Service keeps up to 3 models (`MAX_RESIDENT = 3` in `serve_viewer.py`),
one for each model in the demo. A 4th would push out the one used least recently; the metrics panel
*Multiplexed model unloads per replica* shows when that happens.


All three answers for the same molecule, on top of its crystal: base model (grey), our fine-tune (teal)
and public fine-tune (red), each with its distance from the crystal ligand (dark).

In [ ]:
from aiddd_pipeline import pose_in_crystal_frame

colors = {
    "base": wh.COLORS["of3p2"],
    "ev71-v1": wh.COLORS["of3ray"],
    "ev71-v2": wh.COLORS["of3ft"],
}
poses = []
for model in ("base", "ev71-v1", "ev71-v2"):
    if (
        USE_GPUS
    ):  # your own deployment answers with the raw prediction: move it onto the crystal here
        o = await handle.options(multiplexed_model_id=model).remote(smiles)
        poses.append(
            (
                o["model"]["name"],
                pose_in_crystal_frame(o["cif_path"], compound_data["ref_pdb"]),
                colors[model],
                None,
            )
        )
    else:  # the Service answers with the pose already on the crystal, and its RMSD
        o = requests.post(
            f"{SERVICE_URL}/predict",
            json={"smiles": smiles, "model": model},
            timeout=900,
        ).json()
        if o.get("status") == "failed":
            print(f"{model}: not on the Service yet")
            continue
        poses.append(
            (o["model"]["name"], o["cif"], colors[model], o["answer_key"]["lig_rmsd"])
        )
wh.show_live_overlay(compound_data, poses)

### 6.6 How the Service is built

![The Service app: one request, composed across CPU and GPU deployments](docs/images/serve_composition.svg)

**The usual setup** is a model endpoint, a separate web app, and someone to run the cluster behind them.
Here it is one Python file. One request to the Service uses 3 deployments (`Viewer.predict` in
`serve_viewer.py`):
1. The Viewer prepares the molecule with RDKit, on CPU.
2. It asks OpenFold3 (the pose) and Boltz-2 (the binding strength) at the same time, on GPUs.
3. It scores the pose against the crystal.

**Why Boltz-2 is here at all.** On EV-A71 2A late follow-ons it places only 1% of ligands within 2 Å of
the crystal, so it is not the model you ask for a pose. It is the only model in this app that predicts
binding strength, which is the number the Viewer shows beside the pose. Each deployment answers the
question it is good at, and composition is what lets one request use both.

- **`OF3Pool`** is the 6.1 deployment plus what a shared Service needs: it answers cached poses without
  loading a model, and live predictions take turns on the GPU.
- **Each deployment asks for its own resources and scales on its own.** OF3Pool and Boltz-2 each ask for
  half an L4 and 15 GB of RAM. A live Boltz-2 run uses about 18 GB of a machine's 32 GB, so the two do not
  fit on one machine, and their memory requests make Ray put them on different machines.
- **An existing model joins as it is.** OF3Pool runs OpenFold3 0.4.x in its own Python (`runtime_env`, 6.1),
  next to Boltz-2 in the image's Python.
- **Two kinds of update.** 6.4 and 6.5 changed only `user_config`, so `--in-place` updated the running
  replicas: fast, for iterating. A code or image change is a full deploy, which starts new replicas and can
  send part of the traffic to them first (`--canary-percent`); `anyscale service rollback` reverts a rollout
  while it is in progress. Do not edit Python files between in-place steps: the upload would carry the
  change and restart every replica.

| What Ray Serve and Anyscale give you | Why it matters |
|---|---|
| **Model multiplexing** | The base model and all specialists share the same GPUs; each request picks one |
| **In-place updates** | Add a model by changing `user_config`: no restart, no new machines |
| **Composition** | CPU steps and 2 GPU models in one Python app |
| **A Python per deployment** | OpenFold3 0.4.x and Boltz-2 side by side, no new image |
| **GPUs from every zone** | If one zone has no L4s left, machines start in another |
| **Canary rollouts and rollback** | A new version reaches some users first |


In [ ]:
# Where the Service is now: demo step, the model the viewer uses, models in memory, and what Ray reserved
if SERVICE_URL:
    m = requests.get(f"{SERVICE_URL}/models", timeout=60).json()
    print(
        f"{m.get('step') or 'not on a demo step'} | route: {m['route']} | in memory: {m['resident']} | "
        f"OpenFold3 GPU in {m['zone']} | viewer: {SERVICE_URL}"
    )
    print(
        f"OpenFold3 replica: {m.get('replica_gpus')} GPU, {m.get('replica_memory_reserved_gb')} GB host memory "
        f"reserved, {m.get('replica_memory_gb')} GB used"
    )

### 6.7 Your turn: the shared Service

One person deploys; the whole team uses it, from a browser or from Python. This cell sends the 6.2 molecules to 2 models,
the base model and the one the viewer uses now, 8 at a time, like the whole room at once. Every answer
comes from the cache, so it uses no GPU.

In [ ]:
import concurrent.futures as cf

if not SERVICE_URL:
    print("Skipped. Set AIDDD_SERVICE_URL (README, Workshop format) and re-run Setup.")
else:
    url = SERVICE_URL.rstrip("/")
    route = requests.get(f"{url}/models", timeout=60).json()["route"]
    ids = ["base"] + ([route] if route != "base" else [])

    def ask(job):
        compound, model_id = job
        t0 = time.time()
        r = requests.post(
            f"{url}/predict",
            json={"smiles": DEMO_COMPOUNDS[compound], "model": model_id},
            timeout=900,
        )
        j = r.json() if r.ok else {}
        return {
            "compound": compound,
            "model id": model_id,
            "RMSD (Å)": (j.get("answer_key") or {}).get("lig_rmsd"),
            "source": j.get("source", f"HTTP {r.status_code}"),
            "seconds": round(time.time() - t0, 2),
            "zone": j.get("zone"),
        }

    with cf.ThreadPoolExecutor(8) as ex:
        room = pd.DataFrame(ex.map(ask, [(c, m) for c in DEMO_COMPOUNDS for m in ids]))
    display(
        room.pivot(index="compound", columns="model id", values="RMSD (Å)").round(2)
    )
    display(room)

**What to watch while the room sends requests.** On the Service's page in the Anyscale console, open
the metrics dashboards (Grafana). In the *Serve* dashboard:

| Panel | What you see |
|---|---|
| QPS per deployment | Each Viewer request makes one OF3Pool call and one Boltz2Live call |
| P50 / P90 / P99 latency per deployment | Cached answers stay fast; a live prediction shows as a slow tail |
| Queue size per deployment | Requests waiting for a replica |
| Replicas per deployment | The Viewer runs 2 CPU replicas and can grow to 4 if requests queue; the GPU replicas stay the same |

The *Serve Deployment* dashboard shows multiplexing itself for OF3Pool: *Multiplexed models per replica*,
*Multiplexed model loads per replica* and *Multiplexed model cache hit rate*.

### 6.8 What comes next

Today every answer came from the cache. Real use means new molecules, more targets and more versions.

- **Give the slow model more copies.** A new molecule needs both models: OpenFold3 takes about 25 s,
  Boltz-2 1 to 2 minutes. So requests wait for Boltz-2 (*Queue size per deployment*). Boltz-2 can get
  more replicas by itself, each on its own machine (6.6).
- **More models.** A new model is a new ID, not a new deployment. Each loaded model needs about 3 GB of
  RAM. Add replicas, and Serve sends each ID to a replica that already has it. Today there is one replica,
  so you cannot see Serve choose; with 2 or more, *Multiplexed models per replica* shows it.
- **Keep the current version always ready.** Multiplexing pushes out the model used least recently. To keep
  one version always loaded, give it its own deployment and keep the multiplexed one for older versions.
- **Free idle GPUs.** With `min_replicas: 0`, Serve gives the GPU back when nobody asks. The next request
  then waits for a GPU and for the model to load.
- **GPUs from more than one zone.** A zone is a separate data center in the region. The Service can take
  L4s from any zone, so running out in one zone does not stop it. Traffic
  between zones is a bit slower and costs money per GB. That matters when GPUs work on one job together,
  like training, but not here: each request stays on one replica.


## 7. Summary and next steps

**What we found**

- **Generalization.** On follow-on compounds, Boltz-2 gets 92% of SARS-CoV-2 Mpro poses within 2 Å of the
  crystal, but only 1% for EV-A71 2A (`wh.by_stage` on the stored benchmark). OpenBind-0 gets 91% on
  EV-A71 2A, most likely because similar data was in its training set (notebook 01).
- **Fine-tuning.** 66 fragment-screen structures make OpenFold3 clearly better on follow-on compounds it
  did not see (notebook 02). The result is a specialist for that target.

**What we ran on one Ray cluster**

- **Ray Data.** The full benchmark across 3 campaigns, and 402 new OpenFold3 predictions in 19 minutes on
  12 L4 GPUs.
- **Ray Train.** OpenFold3's own training code on 8 GPUs, with checkpoints and automatic recovery.
- **Ray Serve.** The base model and its specialists in one multiplexed deployment: each request picks one,
  and the most recently used stay in memory. On an Anyscale Service, new versions go in with no restart,
  and a canary rollout is one flag.

**The loop.** Chemists pick the next molecules, the lab makes them and x-rays some, and those crystals
become training data for the next fine-tune. Every round is faster because each step runs on one
platform, not because one step got faster.

The [README](README.md) has the science background and the full runbook. Run the next cell to stop the
presenter's Serve app and free the GPU machines.

In [ ]:
if USE_GPUS:  # only the presenter started a Serve application
    serve.shutdown()
    print("Serve stopped. Idle GPU nodes will be released within a few minutes.")

# Presenter, before the next session: a full deploy (about 20 min, no --in-place) puts the Service on
# step 1 with nothing in memory, so 6.3 to 6.5 show the in-memory list grow again
# !anyscale service deploy -f jobs/demo_1_base.yaml